In [32]:
import unittest

# Task1

In [33]:
#task 1
def hamming_distance(codeword1, codeword2):
    return sum(b1 != b2 for b1, b2 in zip(codeword1, codeword2))

def checking_codewords(codewords, received_data):
    distances = [(codeword, hamming_distance(codeword, received_data)) for codeword in codewords]

    # Find the codeword with the smallest Hamming distance
    distances.sort(key=lambda x: x[1])
    min_dist = distances[0][1]

    # If the received data is already correct
    if min_dist == 0:
        return received_data
    
    # Find all codewords with the same minimum Hamming distance
    closest_codewords = [codeword for codeword, dist in distances if dist == min_dist]

    if len(closest_codewords) == 1:
        return closest_codewords[0]
    else:
        return 'error detected'


In [34]:
#q1
codewords = ['1010111101', '0101110101', '1110101110', '0010110010', '1111000110', '1100101001']
received_data = '1001100101'
print(checking_codewords(codewords, received_data))

0101110101


In [35]:
#q2
codewords = ['1010111101', '0101110101', '1110101110', '0010110010', '1111000110', '1100101001']
received_data = '1000011100'
print(checking_codewords(codewords, received_data))

1010111101


In [36]:
#q3
codewords = ['1010101101', '0101110101', '1110100110', '0000010110', '1101101001']
received_data = '0010110111'
print(checking_codewords(codewords, received_data))

0000010110


# task2

In [37]:
#task 2
def hamming_code_generator(codeword):
    """
    Generates (11,7) Hamming Code by adding parity bits to a 7-bit data word.
    
    :param codeword: A 7-bit binary string (e.g., "1010101")
    :return: An 11-bit Hamming codeword with parity bits
    """
    if len(codeword) != 7:
        raise ValueError("Codeword must be exactly 7 bits long")

    # Positions in the 11-bit Hamming codeword (parity at 1,2,4,8)
    hamming_code = ['0'] * 11

    #Parity bits positions: 1,2,4,8. Indices: 0,1,3,7
    #Data bits positions: 3,5,6,7,9,10,11. Indices: 2,4,5,6,8,9,10
    # Assign data bits to the correct positions (excluding parity bits)
    hamming_code[2] = codeword[0]  # d1
    hamming_code[4] = codeword[1]  # d2
    hamming_code[5] = codeword[2]  # d3
    hamming_code[6] = codeword[3]  # d4
    hamming_code[8] = codeword[4]  # d5
    hamming_code[9] = codeword[5]  # d6
    hamming_code[10] = codeword[6] # d7

    # Calculate parity bits using XOR
    hamming_code[0] = str(int(hamming_code[2]) ^ int(hamming_code[4]) ^ int(hamming_code[6]) ^ int(hamming_code[8]) ^ int(hamming_code[10]))  # p1 (Position 1 = 0001) Positions where the rightmost bit is 1 (1, 3, 5, 7, 9, 11), excluding the parity bit (1) then it becomes, 3,5,7,9,11
    hamming_code[1] = str(int(hamming_code[2]) ^ int(hamming_code[5]) ^ int(hamming_code[6]) ^ int(hamming_code[9]) ^ int(hamming_code[10]))  # p2 (Position 2 = 0010) Positions where the second rightmost bit is 1 (2, 3, 6, 7, 10, 11), excluding the parity bit (1,2) then it becomes, 3,6,7,10,11
    hamming_code[3] = str(int(hamming_code[4]) ^ int(hamming_code[5]) ^ int(hamming_code[6]))  # p3 (Position 4 = 0100) Positions where the third rightmost bit is 1 (4, 5, 6, 7), excluding the parity bit (1,2,4) then it becomes, 5,6,7
    hamming_code[7] = str(int(hamming_code[8]) ^ int(hamming_code[9]) ^ int(hamming_code[10]))  # p4 (Position 8 = 1000) Positions where the fourth rightmost bit is 1 (8, 9, 10, 11), excluding the parity bit (1,2,4,8) then it becomes, 9,10,11

    return ''.join(hamming_code)

# q4
data = "0011101"
encoded_codeword = hamming_code_generator(data)
print("Encoded Codeword:", encoded_codeword)


Encoded Codeword: 11000110101


In [ ]:
def hamming_code_error_detection(received_data):
    """
    Detects and corrects single-bit errors in an (11,7) Hamming Code.
    
    :param received_data: An 11-bit binary string
    :return: The corrected 11-bit codeword or "error detected"
    """
    if len(received_data) != 11:
        raise ValueError("Received data must be exactly 11 bits long")

    # Convert string to list for mutability
    received_bits = list(received_data)

    # Calculate parity check values
    p1 = int(received_bits[0]) ^ int(received_bits[2]) ^ int(received_bits[4]) ^ int(received_bits[6]) ^ int(received_bits[8]) ^ int(received_bits[10]) #positions 1,3,5,7,9,11
    p2 = int(received_bits[1]) ^ int(received_bits[2]) ^ int(received_bits[5]) ^ int(received_bits[6]) ^ int(received_bits[9]) ^ int(received_bits[10]) #positions 2,3,6,7,10,11
    p3 = int(received_bits[3]) ^ int(received_bits[4]) ^ int(received_bits[5]) ^ int(received_bits[6]) #positions 4,5,6,7
    p4 = int(received_bits[7]) ^ int(received_bits[8]) ^ int(received_bits[9]) ^ int(received_bits[10]) #positions 8,9,10,11

    # Calculate error position in binary (p4 p3 p2 p1)
    error_position = (p4 * 8) + (p3 * 4) + (p2 * 2) + (p1 * 1)

    # If error_position is 0, the codeword is correct
    if error_position == 0:
        return 'no error is detected'

    # If error_position is within range, correct the bit
    if 1 <= error_position <= 11:
        print(f"Error detected at position: {error_position}")
        # Flip the incorrect bit
        received_bits[error_position - 1] = '0' if received_bits[error_position - 1] == '1' else '1'
        return ''.join(received_bits)


In [40]:
#q5
received = "11000111101"  # Introduce an error in one bit
corrected_codeword = hamming_code_error_detection(received)
print("Corrected Codeword:", corrected_codeword)


Error detected at position: 8
Corrected Codeword: 11000110101


In [41]:
#q6
received = "00100100110"  # Introduce an error in one bit
corrected_codeword = hamming_code_error_detection(received)
print("Corrected Codeword:", corrected_codeword)


Error detected at position: 6
Corrected Codeword: 00100000110


# Task3